In [1]:
# ============================================================
# INITIALIZE RAG PIPELINE
# ============================================================

from pathlib import Path
import sys

# Notebook:
# rag/Test/Test.ipynb
CURRENT_DIR = Path.cwd()
RAG_DIR = CURRENT_DIR.parent

if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))


# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------

from retrieval.hybrid_retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker
from vector_store.Metadata_filter import MetadataFilter

from context.context_selector import ContextSelector
from context.context_builder import ContextBuilder


# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

CHUNKS_FILE = (
    RAG_DIR
    / "embeddings"
    / "processed"
    / "chunks"
    / "chunks.json"
)


# ------------------------------------------------------------
# INITIALIZE
# ------------------------------------------------------------

print("Initializing Hybrid Retriever...")
hybrid_retriever = HybridRetriever()

print("Initializing Reranker...")
reranker = CrossEncoderReranker()

print("Initializing Metadata Filter...")
metadata_filter = MetadataFilter(
    chunks_file=str(CHUNKS_FILE)
)

print("Initializing Context Selector...")
context_selector = ContextSelector(
    top_k=3
)

print("Initializing Context Builder...")
context_builder = ContextBuilder()


print("\n" + "=" * 70)
print("RAG COMPONENTS READY")
print("=" * 70)

C:\Users\Ahmed\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing Hybrid Retriever...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  85%|████████▍ | 169/199 [00:00<00:00, 1689.12it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1921.99it/s]

Initializing Reranker...
CROSS-ENCODER RERANKER
Loading model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2113.19it/s]

Model loaded successfully.
Initializing Metadata Filter...
Loaded 771 chunks.
Initializing Context Selector...
Initializing Context Builder...

RAG COMPONENTS READY


In [2]:
# ============================================================
# TEST ANSWERABILITY GATE
# ============================================================

from answerability import AnswerabilityGate

gate = AnswerabilityGate(
    min_reranker_score=-4.0,
    min_query_coverage=0.20,
)

TEST_CASES = [
    {
        "question": (
            "How does oxygen concentration affect "
            "flame behavior in microgravity?"
        ),
        "description": "Relevant experimental question",
    },
    {
        "question": (
            "What was the burn time in "
            "experiment CFI-21153-A5?"
        ),
        "description": "Specific experiment lookup",
    },
    {
        "question": (
            "What is the history of the Roman Empire?"
        ),
        "description": "Clearly outside the knowledge base",
    },
    {
        "question": (
            "What was the result of experiment CFI-99999-Z9?"
        ),
        "description": "Likely missing experiment ID",
    },
]

print("=" * 80)
print("ANSWERABILITY GATE TEST")
print("=" * 80)

for test in TEST_CASES:
    question = test["question"]

    # Reuse the retrieval and reranking components.
    hybrid_results = hybrid_retriever.search(
        query=question,
        dense_top_k=20,
        bm25_top_k=20,
        top_k=20,
    )

    enriched_results = metadata_filter.enrich_results(
        hybrid_results
    )

    reranked_results = reranker.rerank(
        query=question,
        candidates=enriched_results,
        top_k=5,
    )

    result = gate.check(
        question=question,
        candidates=reranked_results,
    )

    print("\n" + "-" * 80)
    print("Test:", test["description"])
    print("Question:", question)
    print("Decision:", result["decision"])
    print("Reason:", result["reason"])
    print("Top score:", result["top_score"])
    print(
        "Query coverage:",
        f'{result["query_coverage"]:.2%}'
    )


ANSWERABILITY GATE TEST

CROSS-ENCODER RERANKING
Query: How does oxygen concentration affect flame behavior in microgravity?
Candidates received: 20
Generating reranker scores...



RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 726
Reranker Score: 1.4719
Hybrid Rank: 12
Dense Rank: 2
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 731
Reranker Score: 1.2399
Hybrid Rank: 4
Dense Rank: 3
BM25 Rank: 12

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 737
Reranker Score: 0.6469
Hybrid Rank: 5
Dense Rank: 12
BM25 Rank: 4

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 730
Reranker Score: 0.5824
Hybrid Rank: 1
Dense Rank: 1
BM25 Rank: 6

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 728
Reranker Score: -0.7388
Hybrid Rank: 3
Dense Rank: 7
BM25 Rank: 7

--------------------------------------------------------------------------------
Test: Relevant experimental question
Question: How does oxygen concentration affect flame behavior in microgravity?
Decision: knowledge_base
R


RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 194
Reranker Score: 6.8926
Hybrid Rank: 4
Dense Rank: 2
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 514
Reranker Score: 6.8204
Hybrid Rank: 1
Dense Rank: 5
BM25 Rank: 2

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 534
Reranker Score: 6.7261
Hybrid Rank: 18
Dense Rank: 10
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 466
Reranker Score: 6.6497
Hybrid Rank: 10
Dense Rank: 6
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 515
Reranker Score: 5.8198
Hybrid Rank: 3
Dense Rank: None
BM25 Rank: 1

--------------------------------------------------------------------------------
Test: Specific experiment lookup
Question: What was the burn time in experiment CFI-21153-A5?
Decision: knowledge_base
Reason: Experim


RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 643
Reranker Score: -10.9239
Hybrid Rank: 11
Dense Rank: None
BM25 Rank: 6

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 667
Reranker Score: -10.9253
Hybrid Rank: 17
Dense Rank: None
BM25 Rank: 9

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 663
Reranker Score: -10.9260
Hybrid Rank: 15
Dense Rank: None
BM25 Rank: 8

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 671
Reranker Score: -10.9278
Hybrid Rank: 19
Dense Rank: None
BM25 Rank: 10

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 659
Reranker Score: -10.9285
Hybrid Rank: 13
Dense Rank: None
BM25 Rank: 7

--------------------------------------------------------------------------------
Test: Clearly outside the knowledge base
Question: What is the history of the Roman Empire?
Decision: llm_fallback
Rea


RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 707
Reranker Score: 1.0427
Hybrid Rank: 16
Dense Rank: None
BM25 Rank: 8

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 15
Reranker Score: 1.0207
Hybrid Rank: 18
Dense Rank: None
BM25 Rank: 9

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 19
Reranker Score: 0.9644
Hybrid Rank: 20
Dense Rank: None
BM25 Rank: 10

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 290
Reranker Score: 0.8279
Hybrid Rank: 5
Dense Rank: 3
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 703
Reranker Score: 0.8206
Hybrid Rank: 14
Dense Rank: None
BM25 Rank: 7

--------------------------------------------------------------------------------
Test: Likely missing experiment ID
Question: What was the result of experiment CFI-99999-Z9?
Decision: llm_fallback
Reason: Experim

In [3]:
# ============================================================
# VERIFY SPECIFIC EXPERIMENT RETRIEVAL
# ============================================================

question = "What was the burn time in experiment CFI-21153-A5?"

hybrid_results = hybrid_retriever.search(
    query=question,
    dense_top_k=20,
    bm25_top_k=20,
    top_k=20,
)

enriched_results = metadata_filter.enrich_results(
    hybrid_results
)

reranked_results = reranker.rerank(
    query=question,
    candidates=enriched_results,
    top_k=5,
)

print("=" * 80)
print("EXPERIMENT ID VERIFICATION")
print("=" * 80)

for result in reranked_results:
    metadata = result.get("metadata", {})

    print("\nChunk ID:", result.get("chunk_id"))
    print("Experiment ID:", metadata.get("experiment_id"))
    print("Reranker score:", result.get("reranker_score"))

    text = result.get("text", "")
    print("Text contains CFI-21153-A5:", "CFI-21153-A5" in text.upper())

    if "CFI-21153-A5" in text.upper():
        print("Evidence text:")
        print(text[:1500])


CROSS-ENCODER RERANKING
Query: What was the burn time in experiment CFI-21153-A5?
Candidates received: 20
Generating reranker scores...



RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 194
Reranker Score: 6.8926
Hybrid Rank: 4
Dense Rank: 2
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 514
Reranker Score: 6.8204
Hybrid Rank: 1
Dense Rank: 5
BM25 Rank: 2

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 534
Reranker Score: 6.7261
Hybrid Rank: 18
Dense Rank: 10
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 466
Reranker Score: 6.6497
Hybrid Rank: 10
Dense Rank: 6
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 515
Reranker Score: 5.8198
Hybrid Rank: 3
Dense Rank: None
BM25 Rank: 1
EXPERIMENT ID VERIFICATION

Chunk ID: 194
Experiment ID: CFI-21112-A5
Reranker score: 6.892587661743164
Text contains CFI-21153-A5: False

Chunk ID: 514
Experiment ID: CFI-21153-A5
Reranker score: 6.820363998413086
T

In [4]:

import importlib
import answer.answerability as answerability_module

importlib.reload(answerability_module)

AnswerabilityGate = answerability_module.AnswerabilityGate
gate = AnswerabilityGate()

test_questions = [
    "What was the burn time in experiment CFI-21153-A5?",
    "What was the burn time in experiment CFI-99999-Z9?",
    "What was the peak burner temperature in experiment CFI-21153-A5?",
]

for q in test_questions:
    result = gate.check(q, reranked_results)

    print("Question:", q)
    print("Decision:", result["decision"])
    print("Reason:", result["reason"])
    print("-" * 70)


Question: What was the burn time in experiment CFI-21153-A5?
Decision: knowledge_base
Reason: Experiment CFI-21153-A5 contains a numeric value for burn time.
----------------------------------------------------------------------
Question: What was the burn time in experiment CFI-99999-Z9?
Decision: llm_fallback
Reason: Experiment CFI-99999-Z9 was not found in retrieved evidence.
----------------------------------------------------------------------
Question: What was the peak burner temperature in experiment CFI-21153-A5?
Decision: knowledge_base
Reason: Experiment CFI-21153-A5 contains a numeric value for peak burner temperature.
----------------------------------------------------------------------
